In [9]:
%load_ext autoreload
%autoreload 2

import numpy as np
import polars as pl
import torch
import torch_geometric as pyg
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
import matplotlib.pyplot as plt

from bipartite_gnn.graph_visualizatons import visualize_graph, visualize_embeddings
from baseline_evals.feature_selection import variance_filtering, class_variational_selection
from bipartite_gnn.preprocessing import mrmr_selection
from baseline_evals.knn_eval import knn_eval
from baseline_evals.svm_eval import svm_eval
from baseline_evals.xgboost_eval import xgboost_eval
from baseline_evals.mlp_eval import mlp_eval
from gnn_experiments.mogonet_eval import mogonet_eval

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
mrna = pl.read_csv('MDS_data_preprocessed/mrna.csv')
mirna = pl.read_csv('MDS_data_preprocessed/mirna.csv')
circrna = pl.read_csv('MDS_data_preprocessed/circrna.csv')
te_counts = pl.read_csv('MDS_data_preprocessed/te_counts.csv')
pirna = pl.read_csv('MDS_data_preprocessed/pirna.csv')

annot = pl.read_csv('MDS_data_preprocessed/annotations.csv')

In [4]:
mrna

GENE_NAME,V1884,N58,V630,N60,V1297,NV1428,N82,V940,V2092,V624,V777,V553,V1441,V1921,V538,V1857,V456,NV912,V2089,V1823,V2241,N54,V655,V2110,V839,V125,N83,V637,V712,NV911,V2133,V1742,V1591,V108,V1874,V221,V148,N70,N87,V574,V359,V1337,V883,V1592,V1422,V1708,V1505,V18,V1788,V1776,N84,V1800,V716,N86,V888,V1321,V1279,V1528,V344,N85,V1699,V1456,V1394,V714,V67,V1090,V1860,V406,V1834,V1048,V806,V513,V1565,V1920
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""RILPL1""",187,235,538,244,256,131,89,364,362,164,167,426,128,200,128,276,314,114,120,570,376,407,131,315,139,172,232,44,65,133,93,698,226,60,114,170,203,266,279,352,715,195,273,210,106,139,223,233,184,62,198,464,355,134,474,393,146,153,120,246,345,239,110,92,432,207,373,180,301,404,458,263,384,649
"""RAB4B""",1296,951,1261,856,613,849,749,813,937,1989,882,2094,527,914,948,434,920,454,1798,792,1165,951,685,812,907,710,865,460,786,555,304,1420,1177,1080,2095,636,1026,905,681,1350,1514,1348,887,753,765,1195,1317,860,1033,860,1441,1683,1304,851,1130,1526,1111,1183,1137,682,2000,1259,1390,1370,1374,812,1238,1024,1072,764,801,989,1682,943
"""TIGAR""",1229,184,609,691,173,325,393,227,424,817,243,998,686,332,219,314,366,329,511,567,511,580,342,218,1073,296,356,80,217,240,142,994,488,301,799,304,485,258,277,376,475,504,328,568,133,452,322,453,197,210,288,582,1098,275,362,343,404,309,590,482,793,606,878,630,408,240,340,564,371,404,499,400,508,204
"""DNAH3""",5,0,12,16,12,3,30,2,92,0,10,20,6,0,2,190,3,10,12,6,7,18,134,17,109,29,4,0,9,3,266,5,3,3,6,0,195,8,9,43,4,0,0,0,2,3,8,2,3,4,10,7,23,18,19,97,0,0,8,0,33,0,4,80,6,0,7,3,115,0,3,0,11,22
"""RP11-432M8.3""",2,1,1,0,3,0,2,1,7,0,0,2,1,0,0,17,0,1,0,0,1,0,7,2,1,3,2,0,1,0,8,5,1,1,0,0,0,0,1,0,6,0,4,0,0,0,1,0,0,1,1,0,0,1,2,2,0,2,0,2,3,0,0,0,1,0,3,1,11,0,0,0,0,3
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""CYP4F2""",291,182,157,25,43,114,178,171,14,458,7,8,621,116,177,66,11,57,22,117,119,152,58,474,226,81,120,191,481,26,58,178,50,129,509,59,255,144,203,96,38,32,152,1116,324,370,115,275,18,169,105,57,138,257,84,139,381,30,248,76,62,138,0,237,213,487,189,8,23,0,11,127,281,225
"""TENM1""",131,32,16,11,46,19,20,32,78,4,35,203,48,23,18,250,38,3,31,3,15,22,777,126,57,31,32,6,18,30,108,170,29,173,81,138,90,42,33,39,231,2876,23,7,26,50,23,0,16,42,26,247,58,75,36,93,25,17,232,25,14,7,28,54,11,5,12,107,237,11,19,7,49,34
"""BATF3""",117,67,351,134,70,64,23,84,109,140,262,184,166,33,67,111,52,72,104,125,119,127,76,92,106,124,216,25,181,17,98,131,103,86,259,315,131,147,58,182,223,115,137,48,100,109,150,200,46,54,82,226,343,51,131,50,76,73,37,90,100,128,34,168,210,178,79,60,117,83,472,74,227,39


In [11]:
sample_names = annot['SAMPLE_NAME'].to_list()
sample_names = [name.split('_')[0] for name in sample_names]

assert sample_names == mrna.columns[1:]  == mirna.columns[1:] == circrna.columns[1:] == te_counts.columns[1:]

In [25]:
# prepare pirnas too
# pirna = pl.read_excel("MDS_data/piRNA_counts.xlsx")

# pirna_samples = pirna.columns[1:]

# pirna_samples = [s.split("_")[0] for s in pirna_samples]

# common_samples_w_pirna = list(set(pirna_samples).intersection(set(sample_names)))

# annot_pi = annot.filter(pl.col("sample_ids").is_in(common_samples_w_pirna))

# pirna.columns = ['piRNA'] + pirna_samples
# pirna = pirna.select(['piRNA'] + common_samples_w_pirna)
# pirna_X = pirna.to_numpy()[:, 1:]
# pirna_zeros_mask = pirna_X.sum(axis=1) < 300
# pirna_zeros_idx = np.where(pirna_zeros_mask == False)[0]
# pirna_names = pirna['piRNA'].to_list()
# pirna_names = [name.split('/')[0] for name in pirna_names]
# pirna = pirna.with_columns(piRNA=pl.Series(pirna_names))
# pirna[pirna_zeros_idx].write_csv("MDS_data_preprocessed/pirna.csv")

piRNA,V1565_S17-UMIs,N58_S10-UMIs,V1874_S15-UMIs,V777_S8-UMIs,N80_S20-UMIs,V1788_S3-UMIs,N65_S19-UMIs,V2368_S6-UMIs,N81_S21-UMIs,N59_S18-UMIs,V2286_S2-UMIs,V406_S7-UMIs,V100_S12-UMIs,N82_S1-UMIs,V2133_S13-UMIs,V574_S16-UMIs,V2115_S9-UMIs,V1921_S11-UMIs,V714_S4-UMIs,V637_S14-UMIs,V1742_S5-UMIs,V1744_S2-UMIs,V2248_S3-UMIs,V1428_S4-UMIs,V18_S5-UMIs,V1857_S2-UMIs,V839_S13-UMIs,V912_S21-UMIs,V1048_S9-UMIs,V911_S10-UMIs,V940_S13-UMIs,V681_S21-UMIs,V708_S24-UMIs,N60_S12-UMIs,N70_S11-UMIs,V148_S12-UMIs,…,V1441_S29-UMIs,V1699_S1-UMIs,V1297_S1-UMIs,V1321_S15-UMIs,V1505_S30-UMIs,V1249_S18-UMIs,V1456_S8-UMIs,V1426_S26-UMIs,V1394_S27-UMIs,V1592_S15-UMIs,V1528_S14-UMIs,V1591_S22-UMIs,V833_S6-UMIs,V1708_S27-UMIs,V1800_S16-UMIs,V1776_S23-UMIs,V1823_S11-UMIs,V1775_S15-UMIs,V1834_S3-UMIs,V2378_S19-UMIs,V2414_S10-UMIs,V1860_S4-UMIs,V1884_S28-UMIs,V1920_S7-UMIs,V2322_S18-UMIs,V2311_S18-UMIs,V2291_S17-UMIs,V1957_S6-UMIs,V2092_S17-UMIs,V2284_S16-UMIs,V2278_S20-UMIs,V2110_S7-UMIs,V2179_S9-UMIs,V2147_S8-UMIs,V2224_S19-UMIs,V2089_S1-UMIs,V788_S2-UMIs
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,…,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""hsa_piR_006779…",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""hsa_piR_007653…",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""hsa_piR_009540…",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""hsa_piR_000302…",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,11,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""hsa_piR_000390…",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""hsa_piR_020814…",315,274,230,160,333,154,116,192,213,244,83,147,332,275,94,50,239,168,515,14,122,162,188,111,309,573,18,18,50,59,45,98,70,133,203,138,…,177,108,225,170,193,210,79,42,418,327,144,70,209,246,319,128,88,160,214,139,183,87,94,127,212,78,136,187,67,154,203,222,139,273,161,175,43
"""hsa_piR_020815…",316,458,493,422,279,266,140,417,243,395,220,74,361,450,77,120,715,170,210,12,289,688,336,203,604,137,104,193,270,276,265,202,410,132,225,291,…,340,201,157,1068,389,405,177,89,1005,578,329,314,574,551,458,187,117,421,669,646,183,250,170,131,635,248,170,864,126,293,571,231,224,281,574,183,120
"""hsa_piR_020829…",2732,1064,1346,610,1598,894,848,528,1525,1060,810,565,824,1131,380,1242,833,1493,2428,146,901,1991,378,714,3829,766,226,175,544,380,632,547,312,589,472,1735,…,65,1050,383,193,1198,264,610,112,492,475,1173,1241,330,385,292,375,506,325,687,330,1212,777,665,138,1620,357,621,874,573,661,1069,1170,784,749,1192,921,391


In [12]:
y_d = annot['1 disease'].to_numpy()
y_r = annot['2 risk'].to_numpy()
y_m = annot['3 mutations (SF3B1only_wt)'].to_numpy()

np.unique(y_d, return_counts=True), np.unique(y_r, return_counts=True), np.unique(y_m, return_counts=True)

((array([1, 2]), array([13, 61])),
 (array([0, 1, 2]), array([21, 29, 24])),
 (array([0, 1, 2]), array([48, 14, 12])))

In [5]:
mrna_names = mrna.columns[1:]
mirna_names = mirna.columns[1:]
circrna_names = circrna.columns[1:]
te_names = te_counts.columns[1:]

y_d = annot['1 disease'].to_numpy()
y_r = annot['2 risk'].to_numpy()
y_m = annot['3 mutations (SF3B1only_wt)'].to_numpy()

y = y_d

annotated_samples_mask = y != 0

y = y - 1  # move to 0-indexing

In [58]:
y

array([1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0,
       1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1])

In [6]:
mrna_X = mrna.to_numpy()[:, 1:].T
mirna_X = mirna.to_numpy()[:, 1:].T
circrna_X = circrna.to_numpy()[:, 1:].T
te_X = te_counts.to_numpy()[:, 1:].T

mrna_X = mrna_X[annotated_samples_mask]
mirna_X = mirna_X[annotated_samples_mask]
circrna_X = circrna_X[annotated_samples_mask]
te_X = te_X[annotated_samples_mask]

y = y[annotated_samples_mask]

print(np.unique(y, return_counts=True))

X = np.hstack([mrna_X])

(array([0, 1]), array([13, 61]))


In [9]:
knn_eval(X, y, select_n_features=True)

500
| KNN | 0.91 +/- 0.08 | 0.76 +/- 0.21 | 0.89 +/- 0.10 |
study.best_value=0.8854446886446887, study.best_params={'n_neighbors': 6, 'n_features': 2166}


In [10]:
svm_eval(X, y, select_n_features=True)

Trial 1 / 40
| SVM | 0.81 +/- 0.13 | 0.75 +/- 0.11 | 0.82 +/- 0.11 |
Trial 2 / 40
Trial 3 / 40
| SVM | 0.84 +/- 0.14 | 0.79 +/- 0.15 | 0.85 +/- 0.13 |
Trial 4 / 40
Trial 5 / 40
Trial 6 / 40
Trial 7 / 40
Trial 8 / 40
Trial 9 / 40
Trial 10 / 40
Trial 11 / 40
| SVM | 0.85 +/- 0.11 | 0.79 +/- 0.11 | 0.86 +/- 0.09 |
Trial 12 / 40
Trial 13 / 40
Trial 14 / 40
Trial 15 / 40
Trial 16 / 40
Trial 17 / 40
Trial 18 / 40
Trial 19 / 40
Trial 20 / 40
Trial 21 / 40
Trial 22 / 40
Trial 23 / 40
Trial 24 / 40
Trial 25 / 40
Trial 26 / 40
Trial 27 / 40
Trial 28 / 40
Trial 29 / 40
Trial 30 / 40
Trial 31 / 40
Trial 32 / 40
Trial 33 / 40
Trial 34 / 40
Trial 35 / 40
Trial 36 / 40
Trial 37 / 40
Trial 38 / 40
Trial 39 / 40
Trial 40 / 40
| LIN SVM | 0.85 +/- 0.11 | 0.79 +/- 0.11 | 0.86 +/- 0.09 |
study.best_value=0.8623449121999845, study.best_params={'C': 0.14646961644890166, 'class_weight': 'balanced', 'n_features': 529}


{'acc': 0.8523809523809524,
 'f1_macro': 0.7929974695192087,
 'f1_weighted': 0.8623449121999845,
 'acc_std': 0.10596709368153238,
 'f1_macro_std': 0.1074278261937699,
 'f1_weighted_std': 0.09291001559594489}

In [11]:
xgboost_eval(X, y)

0 / 50
ACC: [0.86666667 0.66666667 0.66666667 0.8        0.85714286]
F1M: [0.71153846 0.4        0.4        0.72049689 0.70833333]
F1W: [0.86666667 0.64       0.64       0.80993789 0.85714286]
| XGBoost | 0.77 +/- 0.09 | 0.59 +/- 0.15 | 0.76 +/- 0.10 |
1 / 50
ACC: [0.73333333 0.66666667 0.66666667 0.8        0.85714286]
F1M: [0.42307692 0.4        0.4        0.72049689 0.70833333]
F1W: [0.73333333 0.64       0.64       0.80993789 0.85714286]
2 / 50
ACC: [0.8        0.8        0.8        0.93333333 0.92857143]
F1M: [0.44444444 0.44444444 0.44444444 0.88       0.81333333]
F1W: [0.77037037 0.71111111 0.71111111 0.928      0.91809524]
| XGBoost | 0.85 +/- 0.06 | 0.61 +/- 0.20 | 0.81 +/- 0.10 |
3 / 50
ACC: [0.93333333 0.93333333 0.93333333 0.73333333 1.        ]
F1M: [0.81481481 0.9068323  0.88       0.7        1.        ]
F1W: [0.92345679 0.93664596 0.928      0.76       1.        ]
| XGBoost | 0.91 +/- 0.09 | 0.86 +/- 0.10 | 0.91 +/- 0.08 |
4 / 50
ACC: [0.86666667 0.8        0.93333333 0.

In [8]:
mlp_eval(X, y, reg_type=None, n_trials=15)

Trial 0 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.875 0.75  0.875 0.75  1.   ]
[0.46666667 0.66666667 0.79487179 0.73333333 1.        ]
[0.81666667 0.75       0.85897436 0.76666667 1.        ]
{'acc': 0.85, 'f1_macro': 0.7323076923076923, 'f1_weighted': 0.8384615384615384, 'acc_std': 0.09354143466934853, 'f1_macro_std': 0.1734637652177653, 'f1_weighted_std': 0.08940963505258638}
Trial 1 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.75  0.75  0.75  0.875 1.   ]
[0.42857143 0.66666667 0.66666667 0.85454545 1.        ]
[0.75       0.75       0.75       0.88181818 1.        ]
Trial 2 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.875 0.75  0.75  0.875 1.   ]
[0.46666667 0.66666667 0.66666667 0.85454545 1.        ]
[0.81666667 0.75       0.75       0.88181818 1.        ]
Trial 3 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.875 0.875 0.75  1.    1.   ]
[0.46666667 0.79487179 0.66666667 1.         1.        ]
[0.81666667 

Exception ignored in: <function _releaseLock at 0x732c003b8ea0>
Traceback (most recent call last):
  File "/home/lubojjan/micromamba/envs/diploma/lib/python3.12/logging/__init__.py", line 243, in _releaseLock
    def _releaseLock():
    
KeyboardInterrupt: 


{'acc': 0.9, 'f1_macro': 0.7856410256410257, 'f1_weighted': 0.8851282051282052, 'acc_std': 0.09354143466934853, 'f1_macro_std': 0.20390403951541392, 'f1_weighted_std': 0.10002169389933914}

In [7]:
mlp_eval(X, y, reg_type='l1', n_trials=15)

Trial 0 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.875      0.625      0.5        0.75       0.85714287]
[0.46666667 0.56363636 0.33333333 0.66666667 0.78787879]
[0.81666667 0.64545455 0.5        0.75       0.87445887]
{'acc': 0.7214285731315613, 'f1_macro': 0.5636363636363637, 'f1_weighted': 0.7173160173160172, 'acc_std': 0.14223077128025008, 'f1_macro_std': 0.15706209986485523, 'f1_weighted_std': 0.13268232310424447}
Trial 1 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.625      0.5        0.5        0.5        0.85714287]
[0.38461538 0.33333333 0.46666667 0.33333333 0.78787879]
[0.67307692 0.5        0.53333333 0.5        0.87445887]
Trial 2 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.875 0.75  0.75  0.75  1.   ]
[0.46666667 0.66666667 0.42857143 0.73333333 1.        ]
[0.81666667 0.75       0.64285714 0.76666667 1.        ]
{'acc': 0.825, 'f1_macro': 0.659047619047619, 'f1_weighted': 0.7952380952380953, 'acc_std': 0.1, 'f1_mac

In [8]:
mlp_eval(X, y, reg_type='inner_mat', n_trials=15)

Trial 0 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
[0.875 0.75  0.875 1.    1.   ]
[0.46666667 0.66666667 0.79487179 1.         1.        ]
[0.81666667 0.75       0.85897436 1.         1.        ]
{'acc': 0.9, 'f1_macro': 0.7856410256410257, 'f1_weighted': 0.8851282051282052, 'acc_std': 0.09354143466934853, 'f1_macro_std': 0.20390403951541392, 'f1_weighted_std': 0.10002169389933914}
Trial 1 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Eval 4 / 5
Eval 5 / 5
Pruning trial after 5 evals, cause 0.6784965034965035 < 0.8851282051282052
[0.625      0.75       0.625      0.625      0.71428573]
[0.38461538 0.66666667 0.56363636 0.56363636 0.65      ]
[0.67307692 0.75       0.64545455 0.64545455 0.75714286]
Trial 2 / 15
Eval 1 / 5
Eval 2 / 5
Eval 3 / 5
Pruning trial after 3 evals, cause 0.6666666666666666 < 0.8851282051282052
[0.5  0.75 0.5  0.   0.  ]
[0.33333333 0.66666667 0.5        0.         0.        ]
[0.58333333 0.75       0.5        0.         0.        ]
Trial 3 / 15
Eval 1 /

In [14]:
val, test = mogonet_eval(
    input_omics={
        "mrna": mrna,
        # "meth": meth,
        # "mirna": mirna,
    },
    n_input_features={
        "mrna": 3000,
        "meth": 1000,
        "mirna": 200,
    },
    y=y,
    params={
        "encoder_hidden_channels": {
            "mrna": 300,
            # "meth": 100,
            # "mirna": 50,
        },
        "encoder_type": "gat",
        "num_classes": 2,
        "graph_style": "threshold",
        "avg_degree": 15,
        # "knn": 15,
        "dropout": 0.0,
        "epochs": 400,
        "log_interval": 50,
        "save_best_model": False,
    }
)

dict_keys(['mrna'])
Fold 1 / 5
Building graph for mrna
0.75 tensor(65.9730)
0.875 tensor(33.5946)
0.9375 tensor(3.8919)
0.90625 tensor(18.)
0.921875 tensor(10.5946)
0.9140625 tensor(13.9189)
0.91015625 tensor(16.1622)
0.912109375 tensor(15.1351)
Isolated samples = 11, avg degree = 15.13513469696045
Epoch: 050:
Train Loss: 0.2412, Train Acc: 0.8983, Train F1 Macro: 0.8520, Train F1 Weighted: 0.9039
Val Loss: 0.5861, Val Acc: 0.5714, Val F1 Macro: 0.3636, Val F1 Weighted: 0.6234
Test Loss: 0.4981, Test Acc: 0.8750, Test F1 Macro: 0.4667, Test F1 Weighted: 0.8167
##################################################
Epoch: 100:
Train Loss: 0.2126, Train Acc: 0.9153, Train F1 Macro: 0.8731, Train F1 Weighted: 0.9190
Val Loss: 0.5303, Val Acc: 0.7143, Val F1 Macro: 0.4167, Val F1 Weighted: 0.7143
Test Loss: 0.4472, Test Acc: 0.8750, Test F1 Macro: 0.4667, Test F1 Weighted: 0.8167
##################################################
Epoch: 150:
Train Loss: 0.2092, Train Acc: 0.9153, Train F1 Macr

In [15]:
print("MOGONET GAT (mrna only) results:")
print(f"| MOGONET GAT (mrna only) val | {val[:, 0].mean():.2f} +/- {val[:, 0].std():.2f} | {val[:, 1].mean():.2f} +/- {val[:, 1].std():.2f} | {val[:, 2].mean():.2f} +/- {val[:, 2].std():.2f} |")
print(f"| MOGONET GAT (mrna only) test | {test[:, 0].mean():.2f} +/- {test[:, 0].std():.2f} | {test[:, 1].mean():.2f} +/- {test[:, 1].std():.2f} | {test[:, 2].mean():.2f} +/- {test[:, 2].std():.2f} |")

MOGONET GAT (mrna only) results:
| MOGONET GAT (mrna only) val | 0.94 +/- 0.13 | 0.93 +/- 0.16 | 0.95 +/- 0.11 |
| MOGONET GAT (mrna only) test | 0.93 +/- 0.07 | 0.89 +/- 0.10 | 0.93 +/- 0.07 |
